# HiPPO: High-Order Polynomial Projection Operators

## 1. Theoretical Foundations & The Memory Bottleneck in Sequences
Recurrent Neural Networks (RNNs) and standard temporal models struggle with long-range memory due to vanishing gradients or exponential forgetting:

$$h_t = \tanh(W h_{t-1} + U x_t)$$

The hidden state $h_t$ is forced to compress infinite historical time series $f_{\le t}$ into a finite vector without a mathematically grounded objective.

### The HiPPO Solution (Gu et al., NeurIPS 2020)
HiPPO formalizes memory as an **online continuous function approximation problem**. Given an input continuous signal $f(t)$, the goal is to maintain the best polynomial approximation of history over time $t$ by projecting onto an orthogonal basis (Shifted Legendre Polynomials):

$$g(t, x) = \sum_{n=0}^{N-1} c_n(t) P_n\left(\frac{x}{t}\right)$$

The coefficients vector $c(t) \in \mathbb{R}^N$ evolves according to a continuous-time linear differential equation:

$$\frac{d}{dt} c(t) = -\frac{1}{t} A c(t) + \frac{1}{t} B f(t)$$

For the **HiPPO-LegS (Shifted Legendre)** measure, the transition matrix $A \in \mathbb{R}^{N \times N}$ and input projection $B \in \mathbb{R}^{N \times 1}$ have closed-form expressions:

$$A_{nk} = -\begin{cases} (2n+1)^{1/2}(2k+1)^{1/2} & \text{if } n > k \\ n+1 & \text{if } n = k \\ 0 & \text{if } n < k \end{cases}, \quad B_n = (2n+1)^{1/2}$$

This exact matrix $A$ forms the mathematical engine powering modern State Space Models including **S4 and Mamba**!

In [ ]:
import numpy as np

def make_hippo_legs_matrix(N):
    """
    Constructs the HiPPO-LegS continuous state matrix A and input vector B.
    """
    A = np.zeros((N, N), dtype=np.float64)
    B = np.zeros((N, 1), dtype=np.float64)
    
    for n in range(N):
        B[n] = np.sqrt(2 * n + 1)
        for k in range(N):
            if n > k:
                A[n, k] = -np.sqrt(2 * n + 1) * np.sqrt(2 * k + 1)
            elif n == k:
                A[n, k] = -(n + 1)
            else:
                A[n, k] = 0.0
    return A, B

def discretize_bilinear(A, B, delta=0.01):
    """
    Discretizes continuous state space (A, B) using Bilinear / Tustin transform:
    A_bar = (I - delta/2 * A)^(-1) * (I + delta/2 * A)
    B_bar = (I - delta/2 * A)^(-1) * delta * B
    """
    N = A.shape[0]
    I = np.eye(N)
    inv_term = np.linalg.inv(I - (delta / 2.0) * A)
    A_bar = inv_term @ (I + (delta / 2.0) * A)
    B_bar = inv_term @ (delta * B)
    return A_bar, B_bar

# Demonstration
N_poly = 8
A, B = make_hippo_legs_matrix(N_poly)
A_bar, B_bar = discretize_bilinear(A, B, delta=0.05)

print(f"HiPPO-LegS Order N = {N_poly}")
print("Continuous Transition Matrix A (Top-Left 4x4):")
print(np.round(A[:4, :4], 2))
print("
Discretized Transition Matrix A_bar Eigenvalues Magnitude:")
eigs = np.abs(np.linalg.eigvals(A_bar))
print(np.round(eigs, 4))
assert np.all(eigs <= 1.0001), "System must be strictly stable!"
print("System stability verified: All discrete eigenvalues lies within unit circle!")

## 2. Signal Memorization & Reconstruction Simulation
Let us verify that passing a complex sinusoidal sequence into the HiPPO discrete recurrent state preserves the complete historical trajectory.

In [ ]:
T_steps = 100
t_axis = np.linspace(0, 4 * np.pi, T_steps)
signal = np.sin(t_axis) + 0.5 * np.cos(3 * t_axis)

# Run discrete HiPPO recurrence
c_state = np.zeros((N_poly, 1))
state_history = []

for t in range(T_steps):
    u = signal[t]
    c_state = A_bar @ c_state + B_bar * u
    state_history.append(c_state.flatten())

state_history = np.array(state_history)
print(f"Signal steps processed: {T_steps}")
print(f"State coefficient shape: {state_history.shape} (Stored in {N_poly} compact dimensions)")
print(f"Final coefficient vector: {np.round(state_history[-1], 3)}")

## 3. High-Yield Interview Questions & Best Practices

### Q1: Why are the eigenvalues of the continuous HiPPO matrix strictly in the left-half complex plane?
**Answer:** For any continuous linear dynamical system $\dot{x} = A x + B u$, stability requires all eigenvalues to have negative real parts ($\operatorname{Re}(\lambda) \le 0$). In HiPPO, because $A$ is derived from continuous projection onto orthogonal Legendre polynomials with an exponential decay or uniform measure, its real spectrum is strictly negative. Discretization maps these eigenvalues strictly inside the unit disk ($|\lambda| \le 1$), guaranteeing bounded state responses and zero gradient explosion over thousands of sequence steps.

### Q2: What is the relationship between HiPPO and the selective mechanism in Mamba?
**Answer:** Structured State Space models (S4) initialize their static state matrices directly using HiPPO-LegS. **Mamba** builds upon S4 by introducing **selectivity**: instead of keeping $B, C,$ and step size $\Delta$ invariant across all inputs, Mamba computes $\Delta(x_t)$, $B(x_t)$, and $C(x_t)$ as data-dependent projections of the current token $x_t$, allowing the model to dynamically remember or forget information while maintaining HiPPO's stable polynomial structure.